# 06 - Smoothed and scaled latent vectors

The 7-day baseline clusters the backward 7-day mean of the daily mean and standard deviation of the price. This notebook gives the Temporal VAE latents the same help, and also tries them scaled. No model is trained: it reads the latents saved by notebook 03, clusters them and saves only the regimes. Every metric is computed afterwards by `scripts/06_regime_metrics.py`, the same way as for the runs of notebooks 03 and 05.

For every trained model in `studies` and every k in `k_values`, k-means runs on these inputs:
- `latents + level`, scaled: the latents and the daily mean and std, with each latent dim z-scored on training days as the price is. (Unscaled, this input is the run of notebook 03 itself.)
- `latents 7d + level 7d`, raw and scaled: the latents and the daily mean and std, all replaced by their backward 7-day mean (the day and the 6 days before it; the first 6 days average the days available).

As in notebook 03, k-means is fitted on training days, assigns every day, and regimes are numbered by ascending mean real price. The regimes are saved in `experiments/temporal_vae/kmeans/latent_smoothing/regimes.csv` (one column per input) and `inputs.csv` (what each column is).

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_context, chronological_split
from src.clustering import level_columns, fit_clusters, order_regimes
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Configuration

`studies` are the study folders of notebook 03 whose trained models are read (a study that does not exist yet is skipped). Each trained model is read once, from the folder named by `trained_in`, so its `artifacts/latents.npy` must be on this computer.

In [ ]:
scalogram_set = "real_price_log"
studies = ["temporal_vae/kmeans/hyperparameters", "temporal_vae/kmeans/lam_beta_grid", "temporal_vae/kmeans/z16"]
k_values = [3, 4, 5]
window = 7
train_ratio, val_ratio = 0.6, 0.2
out_dir = exp.EXPERIMENTS_DIR / "temporal_vae" / "kmeans" / "latent_smoothing"

## 1. Data and trained models

In [ ]:
metadata = pd.read_csv(Path("../data/processed/scalograms") / scalogram_set / "scalogram_metadata.csv",
                       parse_dates=["start_datetime", "end_datetime"])
context = load_context(Path("../data/processed"), metadata)
train_idx, _, _ = chronological_split(len(metadata), train_ratio, val_ratio)
price = context["precio_real_cop_kwh"].to_numpy()

models = {}
for study in studies:
    for path in sorted((exp.EXPERIMENTS_DIR / study).glob("*/config.json")):
        cfg = json.loads(path.read_text())
        if cfg.get("scalogram_set") == scalogram_set and cfg.get("trained_in"):
            models.setdefault(cfg["trained_in"], {"study": cfg["study"], "z_dim": cfg["z_dim"], "lam": cfg["lam"],
                                                  "beta": cfg["beta"], "seed": cfg["seed"]})
models = pd.DataFrame.from_dict(models, orient="index").rename_axis("run_id").reset_index()
models

## 2. Clustering

`zscore` scales columns with the mean and standard deviation of the training days. `backward_mean` is the backward `window`-day mean of each column.

In [ ]:
def zscore(x):
    x = np.asarray(x, dtype=np.float64)
    return (x - x[train_idx].mean(axis=0)) / x[train_idx].std(axis=0)


def backward_mean(x):
    return pd.DataFrame(x).rolling(window, min_periods=1).mean().to_numpy()


level = metadata[list(level_columns(metadata))].to_numpy(dtype=np.float64)
level_day, level_7d = zscore(level), zscore(backward_mean(level))

columns, inputs = {"date": context["date"].dt.date}, []
for _, model in models.iterrows():
    latents = np.load(exp.EXPERIMENTS_DIR / model["run_id"] / "artifacts" / "latents.npy").astype(np.float64)
    latents_7d = backward_mean(latents)
    feature_sets = {
        ("latents + level", "scaled"): np.hstack([zscore(latents), level_day]),
        ("latents 7d + level 7d", "raw"): np.hstack([latents_7d, level_7d]),
        ("latents 7d + level 7d", "scaled"): np.hstack([zscore(latents_7d), level_7d]),
    }
    for (features, latent_scale), x in feature_sets.items():
        for k in k_values:
            name = f"c{len(inputs):03d}"
            columns[name] = order_regimes(fit_clusters(x, train_idx, k, int(model["seed"]))[0], price)
            inputs.append({"column": name, "model": "temporal_vae", **model.to_dict(), "k": k, "features": features,
                           "latents": latent_scale, "smoothed": features != "latents + level"})

out_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(columns).to_csv(out_dir / "regimes.csv", index=False)
pd.DataFrame(inputs).to_csv(out_dir / "inputs.csv", index=False)
print(f"{len(models)} models x {len(feature_sets)} inputs x {len(k_values)} k = {len(inputs)} regime columns")